### **1. Do elastic products generate more revenue or inelastic ones?**

In [0]:
%sql
WITH revenue AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue
    FROM retail
    GROUP BY Description
)

SELECT
    e.elasticity,
    r.total_revenue
FROM elasticity_table e
JOIN revenue r
ON e.Description = r.Description

Databricks visualization. Run in Databricks to view.

###Conclusion:
 Products with near-zero elasticity generate the highest revenue, while highly elastic and positive-elastic products contribute significantly less. Stable demand (low sensitivity) drives revenue, not reactive demand. Hence, Revenue peaks where demand is least sensitive to price.

### **2. Statistics of Elasticity (category wise) vs revenue**


In [0]:
%sql
WITH revenue AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue
    FROM retail
    GROUP BY Description
)

SELECT
    e.category,
    COUNT(*) AS product_count,
    AVG(r.total_revenue) AS avg_revenue,
    SUM(r.total_revenue) AS total_revenue,
    STDDEV(r.total_revenue) AS stddev_revenue,
    MIN(r.total_revenue) AS min_revenue,
    MAX(r.total_revenue) AS max_revenue,
    (MAX(r.total_revenue) - MIN(r.total_revenue)) AS range_revenue
FROM elasticity_table e
JOIN revenue r
ON e.Description = r.Description
GROUP BY e.category

Databricks visualization. Run in Databricks to view.

### The following conclusions are made:-
1. The product count of Anomalous Positive goods are just 657 but the avg revenue pulled is 9282, that is because the average here is biased and affected by fewer high values. So to know it better, I checked standard deviations of each category, which came too high.. 21,075 for this category. Approx 6 Million revenue is generated by this category of just 67 goods. 

2. The inelastic goods stand with 1979 products, standard deviation around 14k which is lesser then anamalous positive category. This large category contributed to 10.4 Million revenue generation.

3. Finally the elastic goods which follows law of demand normally, stands at 1722 product count, a standard deviation of just 4.5k which is way too lesser then above two category. Having this low SD, the minimum revenue pulled is 5 and max is 69k which is double of other two categories. However despite of good looking at first, the total revenue sum is just 4 Million, which ranks this category as 3rd to contribute in revenue.

### **3. Elasticity vs Sales Volume**

In [0]:
%sql
WITH volume AS (
    SELECT
        Description,
        SUM(Quantity) AS total_qty,
        MIN(Quantity) AS min_qty,
        MAX(Quantity) AS max_qty,
        percentile_approx(Quantity, 0.5) AS median_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.category,
    AVG(v.total_qty) AS avg_total_qty,
    AVG(v.min_qty) AS avg_min_qty,
    AVG(v.max_qty) AS avg_max_qty,
    AVG(v.median_qty) AS avg_median_qty
FROM elasticity_table e
JOIN volume v
ON e.Description = v.Description
GROUP BY e.category

The Anomalous positive goods are more in demand then any others. The validation is done by looking at other statistics that is similar accross all 3 categories, so their is no bias at all.

### 

### **4. Elasticity vs Price Level**

In [0]:
%sql
WITH price_stats AS (
    SELECT
        Description,
        AVG(Price) AS avg_price,
        MIN(Price) AS min_price,
        MAX(Price) AS max_price,
        percentile_approx(Price, 0.5) AS median_price
    FROM retail
    GROUP BY Description
)

SELECT
    e.category,
    AVG(p.avg_price) AS avg_price,
    AVG(p.min_price) AS avg_min_price,
    AVG(p.max_price) AS avg_max_price,
    AVG(p.median_price) AS avg_median_price
FROM elasticity_table e
JOIN price_stats p
ON e.Description = p.Description
GROUP BY e.category


The price of anomalous positive goods have a strong bias to pull average, the max price is 40 units that is the reason of such high revenue.  

The max price in the elastic category is 8 units with a demand of 1487, that made very less revenue totally in millions. This shows that _few anomalous products perform extremely well NOT all anomalous products_.  

The inelastic goods stand with avg max price of 22 which made them more normal then the other two because the demand is 3k for them whereas it was 4k for anomalous positive goods. The inelastic goods made 10 million revenue which made them 1st position among all.  

---

### **_Important conclusion made_**

The inelastic goods must be pursued in sales maximization strategy, however the anomalous positive goods must be pursued to maximize sales but only those which made the jump in average, means goods with high demand and high price.

---

### **The risk analysis is as follows:**

- **Some anomalous goods:** High Risk, High Revenue  
- **Inelastic goods overall:** Low Risk, High Revenue  
- **Elastic goods overall:** Low Risk, Low Revenue  